# Prepare entanglement for a teleportation slot

Teleportation needs a shared entangled pair before the unknown qubit can be moved. In this lab, Alice asks the network to prepare one usable pair with Bob before a deadline. The controller chooses a source station, reserves endpoint memories, runs one heralded pair attempt, registers the pair, and decides whether the teleportation job can start.


In [39]:
from __future__ import annotations

from dataclasses import dataclass, field
from math import pow

from simyuj.components import ACTION_TRANSMIT_QUANTUM, PortKind, QuantumChannel
from simyuj.components.memories import (
    MEMORY_ABSORB,
    MemoryAbsorbReport,
    QuantumMemory,
    memory_subsystem_id,
)
from simyuj.components.sources import EntangledPairSource
from simyuj.control import AGENT_REPORT, AgentContext, NodeAgent, SessionRuntime
from simyuj.control.payloads import AgentStart, TimerFired
from simyuj.engine import Timeline
from simyuj.entanglement import EntangledPairRegistry, pair_from_absorbs
from simyuj.network import Network, Node
from simyuj.qstate import SubsystemId
from simyuj.qstate.ops import X, Z, correction_for_bell
from simyuj.qstate.state import fidelity
from simyuj.resources import MemoryRef, ResourceManager
from simyuj.tracing import LogLevel, MemorySink, SimulationLogger


A teleportation request is small on paper: two endpoint memories, a fidelity target, and a deadline. The hard part is making the network produce the pair on time.


In [40]:
JOB_ID = "teleport-job:alice-to-bob"
DEADLINE_TICK = 60_000_000
MIN_FIDELITY = 0.90

ALICE_REF = MemoryRef("alice", "mem", 0)
BOB_REF = MemoryRef("bob", "mem", 0)

print("request:", JOB_ID)
print("deadline tick:", DEADLINE_TICK)
print("minimum fidelity:", MIN_FIDELITY)
print("alice endpoint:", ALICE_REF.key)
print("bob endpoint:", BOB_REF.key)


request: teleport-job:alice-to-bob
deadline tick: 60000000
minimum fidelity: 0.9
alice endpoint: ('alice', 'mem', 0)
bob endpoint: ('bob', 'mem', 0)


The link-layer view from the literature is useful here: treat entanglement generation as a service request, then make the route and memory decisions before the physical attempt starts.


In [41]:
@dataclass(frozen=True)
class FiberSpan:
    length_km: float
    attenuation_db_per_km: float
    insertion_loss_db: float
    fidelity_estimate: float

    @property
    def loss_db(self) -> float:
        return self.length_km * self.attenuation_db_per_km + self.insertion_loss_db

    @property
    def survival(self) -> float:
        return 10 ** (-self.loss_db / 10)

    @property
    def delay_ticks(self) -> int:
        seconds = self.length_km * 1000 / 2.0e8
        return round(seconds * 1_000_000_000_000)


In [42]:
network = Network("teleport_entanglement_service")

for node_id in ("controller", "alice", "bob", "eps_balanced", "eps_cheap"):
    network.add_node(Node(node_id))

print("nodes:", tuple(network.nodes))


nodes: ('controller', 'alice', 'bob', 'eps_balanced', 'eps_cheap')


In [43]:
spans = {
    "balanced-left": ("eps_balanced", "alice", FiberSpan(3.0, 0.18, 0.15, 0.94)),
    "balanced-right": ("eps_balanced", "bob", FiberSpan(3.5, 0.18, 0.15, 0.94)),
    "cheap-left": ("eps_cheap", "alice", FiberSpan(15.0, 0.22, 1.00, 0.88)),
    "cheap-right": ("eps_cheap", "bob", FiberSpan(18.0, 0.22, 1.20, 0.88)),
}

for link_id, (source, target, span) in spans.items():
    network.add_quantum_link(link_id, source, target, channel=span)

print("candidate quantum links:")
for link_id, link in network.quantum_links.items():
    span = link.transport
    print(f"  {link_id}: {link.source_node_id} -> {link.target_node_id}, {span.length_km:.1f} km")


candidate quantum links:
  balanced-left: eps_balanced -> alice, 3.0 km
  balanced-right: eps_balanced -> bob, 3.5 km
  cheap-left: eps_cheap -> alice, 15.0 km
  cheap-right: eps_cheap -> bob, 18.0 km


Each source station needs two routes: one member to Alice and one member to Bob. The controller scores both halves together because teleportation needs the pair, not one lucky photon.


In [44]:
@dataclass(frozen=True)
class CandidateScore:
    source_node_id: str
    left_route: object
    right_route: object
    left_loss_db: float
    right_loss_db: float
    left_survival: float
    right_survival: float
    joint_success: float
    latest_arrival_tick: int
    fidelity_estimate: float
    usable: bool


In [45]:
def spans_for(route):
    return [network.get_link(link_id).transport for link_id in route.link_ids]


def route_loss_db(route) -> float:
    return sum(span.loss_db for span in spans_for(route))


def route_delay_ticks(route) -> int:
    return sum(span.delay_ticks for span in spans_for(route))


def route_fidelity_estimate(route) -> float:
    return min(span.fidelity_estimate for span in spans_for(route))


In [46]:
def score_candidate(source_node_id: str) -> CandidateScore:
    left_route = network.fewest_hops_path(source_node_id, "alice", port_kind=PortKind.QUANTUM)
    right_route = network.fewest_hops_path(source_node_id, "bob", port_kind=PortKind.QUANTUM)
    if left_route is None or right_route is None:
        raise RuntimeError(f"{source_node_id} cannot reach both endpoints")

    left_loss = route_loss_db(left_route)
    right_loss = route_loss_db(right_route)
    left_survival = pow(10, -left_loss / 10)
    right_survival = pow(10, -right_loss / 10)
    joint_success = left_survival * right_survival
    latest_arrival = max(route_delay_ticks(left_route), route_delay_ticks(right_route))
    fidelity = min(route_fidelity_estimate(left_route), route_fidelity_estimate(right_route))

    return CandidateScore(
        source_node_id,
        left_route,
        right_route,
        left_loss,
        right_loss,
        left_survival,
        right_survival,
        joint_success,
        latest_arrival,
        fidelity,
        joint_success >= 0.25 and latest_arrival <= DEADLINE_TICK and fidelity >= MIN_FIDELITY,
    )


In [47]:
candidate_scores = [
    score_candidate("eps_balanced"),
    score_candidate("eps_cheap"),
]

print("source          left dB  right dB  joint success  latest tick  fidelity  usable")
for score in candidate_scores:
    print(
        f"{score.source_node_id:14} "
        f"{score.left_loss_db:7.2f} "
        f"{score.right_loss_db:8.2f} "
        f"{score.joint_success:13.3f} "
        f"{score.latest_arrival_tick:11} "
        f"{score.fidelity_estimate:8.2f} "
        f"{score.usable}"
    )


source          left dB  right dB  joint success  latest tick  fidelity  usable
eps_balanced      0.69     0.78         0.713    17500000     0.94 True
eps_cheap         4.30     5.16         0.113    90000000     0.88 False


In [48]:
usable_scores = [score for score in candidate_scores if score.usable]
selected = sorted(
    usable_scores,
    key=lambda score: (-score.joint_success, score.latest_arrival_tick, score.source_node_id),
)[0]

print("selected source:", selected.source_node_id)
print("left route:", selected.left_route.link_ids)
print("right route:", selected.right_route.link_ids)
print("expected joint success:", round(selected.joint_success, 3))


selected source: eps_balanced
left route: ('balanced-left',)
right route: ('balanced-right',)
expected joint success: 0.713


Before touching hardware, reserve the endpoint memories. A real service should fail early if a needed memory is already busy.


In [49]:
busy_manager = ResourceManager()
busy_manager.register_memory("alice", "mem", num_positions=1)
busy_manager.register_memory("bob", "mem", num_positions=1)
busy_manager.mark_occupied(BOB_REF)

try:
    busy_manager.reserve_memories(
        0,
        {"alice": {"mem": 1}, "bob": {"mem": 1}},
        owner=JOB_ID,
        reservation_id="reservation:busy-test",
    )
except ValueError as exc:
    print("busy memory branch:", exc)


busy memory branch: device 'mem' on node 'bob' has 0 available memory slot(s), but 1 requested


Now build only the selected physical path. The unselected source stays as planning metadata; it does not get devices or runtime wires.


In [50]:
left_span = network.get_link(selected.left_route.link_ids[0]).transport
right_span = network.get_link(selected.right_route.link_ids[0]).transport

source = EntangledPairSource(
    device_id=selected.source_node_id,
    frequency_hz=1e6,
    emission_probability=1.0,
    duration_s=1e-12,
)
left_channel = QuantumChannel(
    channel_id=selected.left_route.link_ids[0],
    length_m=left_span.length_km * 1000,
    attenuation_db_per_km=left_span.attenuation_db_per_km,
    fixed_insertion_loss_db=left_span.insertion_loss_db,
)
right_channel = QuantumChannel(
    channel_id=selected.right_route.link_ids[0],
    length_m=right_span.length_km * 1000,
    attenuation_db_per_km=right_span.attenuation_db_per_km,
    fixed_insertion_loss_db=right_span.insertion_loss_db,
)


In [51]:
alice_memory = QuantumMemory(memory_id="alice.mem", num_positions=1)
bob_memory = QuantumMemory(memory_id="bob.mem", num_positions=1)

print("left channel survival:", round(left_channel.survival_probability, 3))
print("right channel survival:", round(right_channel.survival_probability, 3))
print("left channel delay tick:", left_channel.resolved_delay_ticks)
print("right channel delay tick:", right_channel.resolved_delay_ticks)


left channel survival: 0.853
right channel survival: 0.836
left channel delay tick: 15000000
right channel delay tick: 17500000


The controller is small on purpose. It reserves, starts the source, waits for memory reports, then turns two successful absorbs into one reserved pair record.


In [52]:
REF_BY_MEMORY_ID = {
    "alice.mem": ALICE_REF,
    "bob.mem": BOB_REF,
}
@dataclass(slots=True)
class TeleportPrepAgent(NodeAgent):
    source: EntangledPairSource
    selected: CandidateScore
    request_id: str
    min_fidelity: float
    deadline_tick: int
    reservation_id: str | None = None
    reservation_refs: tuple[tuple[str, str, int], ...] = ()
    source_reports: list[object] = field(default_factory=list)
    absorb_reports: list[MemoryAbsorbReport] = field(default_factory=list)
    pair_id: str | None = None
    pair_state: str | None = None
    bell_label: str | None = None
    ready_for_teleportation: bool = False
    reason: str = "not started"
    ready_time: int | None = None
    def on_start(self, start: AgentStart, ctx: AgentContext) -> None:
        del start
        reserve_slot_and_start(self, ctx)
    def on_report(self, report: object, ctx: AgentContext) -> None:
        if isinstance(report, MemoryAbsorbReport):
            self.absorb_reports.append(report)
            if len(self.absorb_reports) == 2 and not self.ready_for_teleportation:
                complete_pair_if_ready(self, ctx)
            return
        self.source_reports.append(report)
    def on_timer(self, timer: TimerFired, ctx: AgentContext) -> None:
        miss_deadline(self, timer, ctx)


In [53]:
def reserve_slot_and_start(agent: TeleportPrepAgent, ctx: AgentContext) -> None:
    reservation = ctx.resources.reserve_memories(
        ctx.timeline.current_time,
        {"alice": {"mem": 1}, "bob": {"mem": 1}},
        reservation_id="reservation:teleport-slot:1",
        created_at=ctx.timeline.current_time,
        expires_at=agent.deadline_tick,
        metadata=(
            ("request_id", agent.request_id),
            ("source", agent.selected.source_node_id),
            ("left_route", agent.selected.left_route.link_ids),
            ("right_route", agent.selected.right_route.link_ids),
        ),
    )
    committed = ctx.resources.commit(reservation.reservation_id)
    agent.reservation_id = committed.reservation_id
    agent.reservation_refs = committed.memory_ref_keys
    agent.reason = "waiting for two memory absorb reports"

    delay = agent.deadline_tick - ctx.timeline.current_time
    ctx.timers.set("teleport-slot-deadline", delay, correlation_id=agent.request_id)
    agent.source.schedule_start(ctx.timeline)


In [54]:
def finish_decision(agent: TeleportPrepAgent, ctx: AgentContext, reserved, bell) -> None:
    agent.pair_id = reserved.pair_id
    agent.pair_state = reserved.state.value
    agent.bell_label = bell.label
    agent.ready_time = ctx.timeline.current_time
    agent.ready_for_teleportation = (
        reserved.fidelity is not None
        and reserved.fidelity >= agent.min_fidelity
        and ctx.timeline.current_time <= agent.deadline_tick
    )
    agent.reason = (
        "reserved pair meets fidelity and deadline"
        if agent.ready_for_teleportation
        else "pair arrived but missed the service target"
    )


In [55]:
def complete_pair_if_ready(agent: TeleportPrepAgent, ctx: AgentContext) -> None:
    alice_report = next(report for report in agent.absorb_reports if report.memory_id == "alice.mem")
    bob_report = next(report for report in agent.absorb_reports if report.memory_id == "bob.mem")

    for report in (alice_report, bob_report):
        ctx.resources.mark_absorb_report(report, REF_BY_MEMORY_ID[report.memory_id])

    pair = pair_from_absorbs(
        "pair:teleport-slot:1",
        ALICE_REF,
        alice_report,
        BOB_REF,
        bob_report,
        fidelity=agent.selected.fidelity_estimate,
        created_at=ctx.timeline.current_time,
        expires_at=agent.deadline_tick,
        generation_link_id="+".join(agent.selected.left_route.link_ids + agent.selected.right_route.link_ids),
        left_memory_id="alice.mem",
        right_memory_id="bob.mem",
        metadata=(("request_id", agent.request_id), ("selected_source", agent.selected.source_node_id)),
    )
    reserved = ctx.pairs.reserve(ctx.pairs.register(pair).pair_id)
    bell = ctx.timeline.qstate.measure_bell(
        targets=(memory_subsystem_id("alice.mem", 0), memory_subsystem_id("bob.mem", 0)),
        collapse=False,
    )
    finish_decision(agent, ctx, reserved, bell)


In [56]:
def miss_deadline(agent: TeleportPrepAgent, timer: TimerFired, ctx: AgentContext) -> None:
    if timer.timer_id != "teleport-slot-deadline" or agent.ready_for_teleportation:
        return
    if agent.reservation_id is not None:
        ctx.resources.release(agent.reservation_id)
    agent.reason = f"deadline reached with {len(agent.absorb_reports)} memory report(s)"


Attach the selected source, channels, memories, and controller to the network, then wire the actual event path.


In [57]:
agent = TeleportPrepAgent(
    agent_id="teleport-prep-controller",
    node_id="controller",
    source=source,
    selected=selected,
    request_id=JOB_ID,
    min_fidelity=MIN_FIDELITY,
    deadline_tick=DEADLINE_TICK,
)

network.get_node("controller").add_agent(agent)
network.get_node(selected.source_node_id).add_device("source", source)
network.get_node(selected.source_node_id).add_device("left_channel", left_channel)
network.get_node(selected.source_node_id).add_device("right_channel", right_channel)
network.get_node("alice").add_device("mem", alice_memory)
network.get_node("bob").add_device("mem", bob_memory)

print("runtime source:", selected.source_node_id)
print("runtime endpoint memories:", ALICE_REF.key, BOB_REF.key)


runtime source: eps_balanced
runtime endpoint memories: ('alice', 'mem', 0) ('bob', 'mem', 0)


In [58]:
network.wire_ports(
    "source-left-to-channel",
    source.left_output_port,
    left_channel.input_port,
    target_action=ACTION_TRANSMIT_QUANTUM,
)
network.wire_ports(
    "channel-left-to-alice-memory",
    left_channel.output_port,
    alice_memory.input_port,
    target_action=MEMORY_ABSORB,
)
network.wire_ports(
    "source-right-to-channel",
    source.right_output_port,
    right_channel.input_port,
    target_action=ACTION_TRANSMIT_QUANTUM,
)
network.wire_ports(
    "channel-right-to-bob-memory",
    right_channel.output_port,
    bob_memory.input_port,
    target_action=MEMORY_ABSORB,
)


PortConnection(connection_id='channel-right-to-bob-memory', source_port=Port(name='out', owner_id='balanced-right', port_kind=<PortKind.QUANTUM: 'quantum'>, direction=<PortDirection.EGRESS: 'egress'>), target_port=Port(name='in', owner_id='bob.mem', port_kind=<PortKind.QUANTUM: 'quantum'>, direction=<PortDirection.INGRESS: 'ingress'>), target_action='memory_absorb')

In [59]:
network.wire_ports(
    "alice-memory-to-controller",
    alice_memory.notice_port,
    agent.reports.port("alice_memory"),
    target_action=AGENT_REPORT,
)
network.wire_ports(
    "bob-memory-to-controller",
    bob_memory.notice_port,
    agent.reports.port("bob_memory"),
    target_action=AGENT_REPORT,
)
network.wire_ports(
    "source-report-to-controller",
    source.report_port,
    agent.reports.port("source"),
    target_action=AGENT_REPORT,
)

print("runtime wires:", tuple(network.wires))


runtime wires: ('source-left-to-channel', 'channel-left-to-alice-memory', 'source-right-to-channel', 'channel-right-to-bob-memory', 'alice-memory-to-controller', 'bob-memory-to-controller', 'source-report-to-controller')


`ResourceManager.from_network` scans the actual `QuantumMemory` devices. At this point the physical memories are empty and the resource slots are free.


In [60]:
resources = ResourceManager.from_network(network)
pairs = EntangledPairRegistry()

print("resource slots before run:")
for ref in resources.registered_memories():
    print(" ", ref.key, resources.get_slot(ref).state.value)
print("pair registry before run:", pairs.all_pairs())


resource slots before run:
  ('alice', 'mem', 0) free
  ('bob', 'mem', 0) free
pair registry before run: ()


In [61]:
sink = MemorySink()
logger = SimulationLogger(
    level=LogLevel.DEBUG,
    sinks=[sink],
    session_id="teleport-capstone",
)
timeline = Timeline(master_seed=19, logger=logger)

runtime = SessionRuntime(
    timeline=timeline,
    network=network,
    resource_manager=resources,
    pair_registry=pairs,
    session_id="teleport-capstone",
)

teleport_rng = timeline.rng("logical_teleportation", "alice_bell")
print("teleportation RNG stream declared before runtime starts")


teleportation RNG stream declared before runtime starts


In [62]:
runtime.run()

print("timeline current tick:", timeline.current_time)
print("events scheduled:", timeline.events_scheduled)
print("events executed:", timeline.events_executed)


timeline current tick: 60000000
events scheduled: 11
events executed: 11


The controller should now know whether the teleportation slot has a reserved pair ready for use.


In [63]:
print("selected source:", selected.source_node_id)
print("route links:", selected.left_route.link_ids + selected.right_route.link_ids)
print("expected joint success:", round(selected.joint_success, 3))
print("reservation id:", agent.reservation_id)
print("reservation refs:", agent.reservation_refs)
print("ready time:", agent.ready_time)


selected source: eps_balanced
route links: ('balanced-left', 'balanced-right')
expected joint success: 0.713
reservation id: reservation:teleport-slot:1
reservation refs: (('alice', 'mem', 0), ('bob', 'mem', 0))
ready time: 17500000


In [64]:
print("left channel delivered/lost:", left_channel.delivered_count, left_channel.lost_count)
print("right channel delivered/lost:", right_channel.delivered_count, right_channel.lost_count)
print("memory reports:", [(r.memory_id, r.position, r.success) for r in agent.absorb_reports])
print("source reports:", [getattr(r, "report_id", None) for r in agent.source_reports])


left channel delivered/lost: 1 0
right channel delivered/lost: 1 0
memory reports: [('alice.mem', 0, True), ('bob.mem', 0, True)]
source reports: ['eps_balanced:prep:1']


In [65]:
print("resource slots after run:")
for ref in resources.registered_memories():
    print(" ", ref.key, resources.get_slot(ref).state.value)

print("pair registry after run:")
for pair in pairs.all_pairs():
    print(" ", pair.pair_id, pair.state.value, pair.fidelity, pair.memory_ref_keys)


resource slots after run:
  ('alice', 'mem', 0) occupied
  ('bob', 'mem', 0) occupied
pair registry after run:
  pair:teleport-slot:1 reserved 0.94 (('alice', 'mem', 0), ('bob', 'mem', 0))


In [66]:
pair = pairs.get(agent.pair_id) if agent.pair_id is not None else None

print("pair id:", agent.pair_id)
print("pair state:", agent.pair_state)
print("fidelity estimate:", None if pair is None else pair.fidelity)
print("qstate Bell label:", agent.bell_label)
print("usable for teleportation:", agent.ready_for_teleportation)
print("reason:", agent.reason)


pair id: pair:teleport-slot:1
pair state: reserved
fidelity estimate: 0.94
qstate Bell label: phi+
usable for teleportation: True
reason: reserved pair meets fidelity and deadline


A compact trace tells the story without dumping every event object. Look for runtime start, source emission, channel forwarding, memory absorbs, and the final pair decision.


In [67]:
category_counts = {}
for record in sink.records:
    category_counts[record.category] = category_counts.get(record.category, 0) + 1

print("trace category counts:")
for category, count in sorted(category_counts.items()):
    print(f"  {count:2}  {category}")


trace category counts:
   2  components.channels.quantum.ready
   2  components.channels.quantum.signal_forwarded
   2  components.memories.quantum_memory.absorb
   2  components.memories.quantum_memory.ready
   1  components.sources.entangled_pair_source.emit
   1  components.sources.entangled_pair_source.start


In [68]:
interesting = [
    "components.sources.entangled_pair_source.start",
    "components.sources.entangled_pair_source.emit",
    "components.channels.quantum.signal_forwarded",
    "components.channels.quantum.signal_lost",
    "components.memories.quantum_memory.absorb",
]

print("important trace records:")
for record in sink.records:
    if record.category in interesting:
        print(f"  t={record.sim_time:>8}  {record.category}  {record.message}")


important trace records:
  t=       0  components.sources.entangled_pair_source.start  source started
  t=       0  components.sources.entangled_pair_source.emit  pair emitted
  t=       0  components.channels.quantum.signal_forwarded  quantum signal forwarded
  t=       0  components.channels.quantum.signal_forwarded  quantum signal forwarded
  t=15000000  components.memories.quantum_memory.absorb  memory absorb reported
  t=17500000  components.memories.quantum_memory.absorb  memory absorb reported


In [69]:
print("ready_for_teleportation_slot:", agent.ready_for_teleportation)
print("reason:", agent.reason)
print("reserved pair id:", agent.pair_id)
print("memory endpoints:", ALICE_REF.key, BOB_REF.key)
print("handoff: the next cells consume this reserved pair.")


ready_for_teleportation_slot: True
reason: reserved pair meets fidelity and deadline
reserved pair id: pair:teleport-slot:1
memory endpoints: ('alice', 'mem', 0) ('bob', 'mem', 0)
handoff: the next cells consume this reserved pair.


Try changing one number: raise `MIN_FIDELITY`, shorten `DEADLINE_TICK`, or increase one span length. The useful question is not “did a photon move?” It is “can this request safely start teleportation now?”


## Consume the pair

Now finish the story. The entanglement service has done its job; the teleportation protocol can use the reserved pair.


In [70]:
print("What this second half does:")
print("  physical distribution already happened through source, channels, and memories")
print("  Alice's Bell measurement is a logical qstate operation")
print("  Bob's correction is a logical qstate operation")
print("  the classical correction message is represented as data")
print("  Bob's memory stays occupied because it holds the teleported output")


What this second half does:
  physical distribution already happened through source, channels, and memories
  Alice's Bell measurement is a logical qstate operation
  Bob's correction is a logical qstate operation
  the classical correction message is represented as data
  Bob's memory stays occupied because it holds the teleported output


Alice's payload is prepared as a known test state so we can check the result. In a real teleportation request, the protocol would not need to know this state.


In [71]:
PAYLOAD = SubsystemId("alice.payload.qubit")
ALICE_HALF_QUBIT = memory_subsystem_id("alice.mem", 0)
BOB_OUTPUT_QUBIT = memory_subsystem_id("bob.mem", 0)

payload_ref = timeline.qstate.prepare(
    "|+i>",
    subsystems=(PAYLOAD,),
    meta=(("request_id", JOB_ID), ("role", "payload_to_teleport")),
)
original_payload = timeline.qstate.get(payload_ref)

print("payload subsystem:", PAYLOAD)
print("payload state:", "|+i>")
print("payload state ref:", payload_ref)
print("Bob output subsystem:", BOB_OUTPUT_QUBIT)


payload subsystem: alice.payload.qubit
payload state: |+i>
payload state ref: 1
Bob output subsystem: memory:bob.mem:position:0


Alice performs a Bell measurement on the payload and her half of the reserved pair. The result is two classical correction bits for Bob.


In [72]:
alice_bell = timeline.qstate.measure_bell(
    targets=(PAYLOAD, ALICE_HALF_QUBIT),
    rng=teleport_rng,
    collapse=True,
)
x_bit, z_bit = correction_for_bell(alice_bell.label)

classical_correction = {
    "from": "alice",
    "to": "bob",
    "bell_label": alice_bell.label,
    "x": x_bit,
    "z": z_bit,
}

print("Alice Bell outcome:", alice_bell.label)
print("Bell probability:", alice_bell.probability)
print("classical correction:", classical_correction)


Alice Bell outcome: psi-
Bell probability: 0.25
classical correction: {'from': 'alice', 'to': 'bob', 'bell_label': 'psi-', 'x': 1, 'z': 1}


Bob applies the correction to his memory. The order of `X` and `Z` only changes a global phase for the double-correction case, so this lab uses one fixed order.


In [73]:
applied_corrections = []
if z_bit:
    timeline.qstate.apply(Z, targets=(BOB_OUTPUT_QUBIT,))
    applied_corrections.append("Z")
if x_bit:
    timeline.qstate.apply(X, targets=(BOB_OUTPUT_QUBIT,))
    applied_corrections.append("X")

print("Bob applied:", applied_corrections or ["no correction"] )
print("Bob state ref after correction:", timeline.qstate.state_of(BOB_OUTPUT_QUBIT))


Bob applied: ['Z', 'X']
Bob state ref after correction: 2


To check the handoff, drop Alice's measured systems from the qstate record. What remains should match the original payload state at Bob.


In [74]:
bob_output_ref = timeline.qstate.discard(targets=(PAYLOAD, ALICE_HALF_QUBIT))
bob_output = timeline.qstate.get(bob_output_ref)
teleportation_fidelity = fidelity(original_payload, bob_output)
teleportation_complete = teleportation_fidelity > 0.999999

print("Bob output state ref:", bob_output_ref)
print("Bob output layout:", timeline.qstate.record(bob_output_ref).layout.subsystems)
print("teleportation fidelity:", round(teleportation_fidelity, 12))
print("teleportation complete:", teleportation_complete)


Bob output state ref: 2
Bob output layout: (SubsystemId(name='memory:bob.mem:position:0'),)
teleportation fidelity: 1.0
teleportation complete: True


The entangled pair is now spent. Alice's pair memory was measured; Bob's memory remains occupied by the teleported output.


In [75]:
consumed_pair = pairs.consume(agent.pair_id)
resources.mark_consumed(ALICE_REF)

print("pair state after consume:", consumed_pair.state.value)
print("Alice slot after teleportation:", resources.get_slot(ALICE_REF).state.value)
print("Bob slot after teleportation:", resources.get_slot(BOB_REF).state.value)
print("active pairs left:", [pair.pair_id for pair in pairs.active_pairs()])


pair state after consume: consumed
Alice slot after teleportation: consumed
Bob slot after teleportation: occupied
active pairs left: []


In [76]:
print("teleportation_complete:", teleportation_complete)
print("request:", JOB_ID)
print("Bell outcome sent to Bob:", alice_bell.label)
print("correction bits:", {"x": x_bit, "z": z_bit})
print("output memory:", BOB_REF.key)
print("pair registry state:", pairs.get(agent.pair_id).state.value)
print("Bob keeps the output qubit in memory.")


teleportation_complete: True
request: teleport-job:alice-to-bob
Bell outcome sent to Bob: psi-
correction bits: {'x': 1, 'z': 1}
output memory: ('bob', 'mem', 0)
pair registry state: consumed
Bob keeps the output qubit in memory.
